# Azure AD OAuth — On-Behalf-Of (User Delegation) Flow

This notebook sets up Snowflake External OAuth so that your FastAPI app can act **on behalf of an Azure AD user**. The user's identity flows through to Snowflake — they get their own roles and audit trail.

---

## How It Works

1. User signs into your app via Azure AD (authorization code flow)
2. Your app exchanges the user's token for a Snowflake-scoped token (On-Behalf-Of flow)
3. Your app connects to Snowflake using that token
4. Snowflake maps the token's `upn` claim → Snowflake user's `login_name`

---

## Azure AD Setup Steps (Portal)

Perform these steps in the **Azure Portal → Microsoft Entra ID → App Registrations** before running the SQL below.

### Step 1: Register the Snowflake Resource App

1. **App Registrations → New Registration**
   - Name: `Snowflake OAuth Resource`
   - Supported account types: Single tenant
2. **Expose an API**
   - Set Application ID URI: `api://<resource-app-client-id>`
   - Add a scope:
     - Name: `session:scope:SNOWFLAKE_API_ROLE`
     - Who can consent: Admins and users
     - Admin consent display name: `Access Snowflake as API Role`
     - State: Enabled

### Step 2: Register the Client App (your FastAPI backend)

1. **App Registrations → New Registration**
   - Name: `Snowflake OAuth Client`
   - Redirect URI: `http://localhost:8000/callback` (or your app's callback URL)
2. **Certificates & secrets → New client secret** → copy the value
3. **API Permissions → Add Permission → My APIs → Snowflake OAuth Resource**
   - Select **Delegated permissions**
   - Check `session:scope:SNOWFLAKE_API_ROLE`
   - Click **Grant Admin Consent**

### Step 3: Collect Values

| Value | Where |
|-------|-------|
| Tenant ID | App Registration → Overview |
| Resource App Client ID | Snowflake OAuth Resource → Overview → Application (client) ID |
| Client App Client ID | Snowflake OAuth Client → Overview → Application (client) ID |
| Client Secret | From Step 2 |
| Issuer | `https://sts.windows.net/<tenant-id>/` |
| JWS Keys URL | `https://login.microsoftonline.com/<tenant-id>/discovery/v2.0/keys` |
| Token Endpoint | `https://login.microsoftonline.com/<tenant-id>/oauth2/v2.0/token` |

---

## Snowflake Setup (run cells below)

In [ ]:
%%sql -r use_role
USE ROLE ACCOUNTADMIN;

### Create the Security Integration

Key differences from client credentials flow:
- `EXTERNAL_OAUTH_TOKEN_USER_MAPPING_CLAIM = 'upn'` — maps the user's Azure AD UPN to a Snowflake user
- `EXTERNAL_OAUTH_SNOWFLAKE_USER_MAPPING_ATTRIBUTE = 'login_name'` — matches against Snowflake user's login_name
- The token carries the user's identity, not the application's

In [ ]:
%%sql -r create_integration
-- Drop existing integration with same issuer
DROP SECURITY INTEGRATION IF EXISTS AZURE_ENTRA_OAUTH;

CREATE OR REPLACE SECURITY INTEGRATION AZURE_ENTRA_OAUTH_OBO
  TYPE = EXTERNAL_OAUTH
  ENABLED = TRUE
  EXTERNAL_OAUTH_TYPE = AZURE
  EXTERNAL_OAUTH_ISSUER = 'https://sts.windows.net/a66a44bf-bf04-4606-839d-3f956853233b/'
  EXTERNAL_OAUTH_JWS_KEYS_URL = 'https://login.microsoftonline.com/a66a44bf-bf04-4606-839d-3f956853233b/discovery/v2.0/keys'
  EXTERNAL_OAUTH_AUDIENCE_LIST = ('api://28c90a4e-4a96-4f78-ab0e-171bd1a984ba')
  EXTERNAL_OAUTH_TOKEN_USER_MAPPING_CLAIM = 'email'
  EXTERNAL_OAUTH_SNOWFLAKE_USER_MAPPING_ATTRIBUTE = 'login_name'
  EXTERNAL_OAUTH_ANY_ROLE_MODE = 'DISABLE';

### Create Role and Grant Privileges

The role name must match the scope defined in Azure AD: `session:scope:SNOWFLAKE_API_ROLE` → role `SNOWFLAKE_API_ROLE`

In [ ]:
%%sql -r create_role
CREATE ROLE IF NOT EXISTS SNOWFLAKE_API_ROLE;

In [ ]:
%%sql -r grant_privileges
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE SNOWFLAKE_API_ROLE;
GRANT USAGE ON DATABASE "SNOWFLAKE_SAMPLE_Apps" TO ROLE SNOWFLAKE_API_ROLE;
GRANT USAGE ON SCHEMA "SNOWFLAKE_SAMPLE_Apps".PUBLIC TO ROLE SNOWFLAKE_API_ROLE;
GRANT SELECT ON ALL TABLES IN SCHEMA "SNOWFLAKE_SAMPLE_Apps".PUBLIC TO ROLE SNOWFLAKE_API_ROLE;
GRANT SELECT ON FUTURE TABLES IN SCHEMA "SNOWFLAKE_SAMPLE_Apps".PUBLIC TO ROLE SNOWFLAKE_API_ROLE;

### Create Snowflake User Mapped to Azure AD User

The `LOGIN_NAME` must match the `upn` claim in the Azure AD token (the user's email/UPN).

Create one Snowflake user per Azure AD user who needs access.

In [ ]:
%%sql -r create_user
-- Replace with your actual Azure AD user's UPN (email)
CREATE USER IF NOT EXISTS AZURE_USER_2
  LOGIN_NAME = 'manish.mishra026@outlook.com'
  DISPLAY_NAME = 'Azure AD User 2'
  DEFAULT_ROLE = SNOWFLAKE_API_ROLE
  DEFAULT_WAREHOUSE = COMPUTE_WH
  MUST_CHANGE_PASSWORD = FALSE;

In [ ]:
%%sql -r grant_role
GRANT ROLE SNOWFLAKE_API_ROLE TO USER AZURE_USER_2;

### Verify the Integration

In [ ]:
%%sql -r desc_integration
DESC SECURITY INTEGRATION AZURE_ENTRA_OAUTH_OBO;

---

## Python FastAPI Code (On-Behalf-Of Flow)

The flow has two stages:
1. **Frontend/User** authenticates with Azure AD and gets an authorization code
2. **Backend (FastAPI)** exchanges code → user token → OBO token → Snowflake connection

In [ ]:
# Reference implementation — On-Behalf-Of Flow with FastAPI
# This cell is for documentation; run it in your FastAPI project, not here.

fastapi_code = '''
import httpx
import snowflake.connector
from fastapi import FastAPI, Depends, HTTPException
from fastapi.responses import RedirectResponse

app = FastAPI()

# --- Configuration ---
TENANT_ID = "<YOUR_TENANT_ID>"
CLIENT_ID = "<YOUR_CLIENT_APP_ID>"        # FastAPI client app
CLIENT_SECRET = "<YOUR_CLIENT_SECRET>"
RESOURCE_APP_ID = "<YOUR_RESOURCE_APP_CLIENT_ID>"  # Snowflake resource app
REDIRECT_URI = "http://localhost:8000/callback"
SCOPE = f"api://{RESOURCE_APP_ID}/session:scope:SNOWFLAKE_API_ROLE"
SNOWFLAKE_ACCOUNT = "kn89629"

AUTHORIZE_URL = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/authorize"
TOKEN_URL = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token"


# --- Step 1: Redirect user to Azure AD login ---
@app.get("/login")
def login():
    params = {
        "client_id": CLIENT_ID,
        "response_type": "code",
        "redirect_uri": REDIRECT_URI,
        "scope": f"openid profile {SCOPE}",
        "response_mode": "query",
    }
    url = f"{AUTHORIZE_URL}?" + "&".join(f"{k}={v}" for k, v in params.items())
    return RedirectResponse(url)


# --- Step 2: Handle callback, exchange code for tokens ---
@app.get("/callback")
async def callback(code: str):
    # Exchange authorization code for user token
    async with httpx.AsyncClient() as client:
        token_resp = await client.post(TOKEN_URL, data={
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "grant_type": "authorization_code",
            "code": code,
            "redirect_uri": REDIRECT_URI,
            "scope": f"openid profile {SCOPE}",
        })
    
    if token_resp.status_code != 200:
        raise HTTPException(status_code=401, detail="Token exchange failed")
    
    access_token = token_resp.json()["access_token"]
    return {"message": "Authenticated", "token_preview": access_token[:20] + "..."}


# --- Step 3: Use the token to query Snowflake ---
async def get_snowflake_token_from_code(code: str) -> str:
    """Exchange auth code for access token scoped to Snowflake."""
    async with httpx.AsyncClient() as client:
        resp = await client.post(TOKEN_URL, data={
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "grant_type": "authorization_code",
            "code": code,
            "redirect_uri": REDIRECT_URI,
            "scope": SCOPE,
        })
    return resp.json()["access_token"]


def get_snowflake_connection(access_token: str):
    """Connect to Snowflake using the user\'s delegated token."""
    return snowflake.connector.connect(
        account=SNOWFLAKE_ACCOUNT,
        authenticator="oauth",
        token=access_token,
        warehouse="COMPUTE_WH",
        database="SNOWFLAKE_SAMPLE_Apps",
        schema="PUBLIC",
    )


@app.get("/employees")
async def get_employees(token: str):
    """Query employees using the user\'s delegated access."""
    conn = get_snowflake_connection(token)
    try:
        cur = conn.cursor()
        cur.execute("SELECT * FROM EMPLOYEES")
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]
        return {"columns": columns, "data": [dict(zip(columns, row)) for row in rows]}
    finally:
        conn.close()
'''

print(fastapi_code)

---

## Troubleshooting Checklist

| Issue | Fix |
|-------|-----|
| `Incorrect username or password` | Snowflake user's `LOGIN_NAME` doesn't match the `upn` claim in the token. Check with: decode the JWT at jwt.ms and compare `upn` value to `LOGIN_NAME` |
| `Role not listed in Access Token` | The scope in your token request must include `session:scope:<ROLE_NAME>`. Verify the scope is `api://<resource-id>/session:scope:SNOWFLAKE_API_ROLE` |
| `Integration not found` | Run `SHOW SECURITY INTEGRATIONS` to verify the integration exists |
| `Token expired` | Access tokens are short-lived (typically 1hr). Use refresh tokens to get new ones |
| `AADSTS65001: consent required` | Admin consent hasn't been granted for the delegated permission. Go to Azure Portal → API Permissions → Grant Admin Consent |

## Key Differences: Client Credentials vs On-Behalf-Of

| Aspect | Client Credentials | On-Behalf-Of |
|--------|-------------------|---------------|
| Token claim for user mapping | `appid` (app ID) | `upn` (user email) |
| Snowflake user represents | The application | The actual human user |
| Audit trail | Shows service account | Shows individual user |
| Role determination | Fixed to service user's role | Based on user's granted roles |
| Scope type in Azure AD | Application permissions | Delegated permissions |
| Grant type | `client_credentials` | `authorization_code` |